# 00 — Collecte des données historiques DE40 5min

**Objectif** : récupérer le maximum de données historiques disponibles pour l'epic `DE40` (DAX 40) à la résolution `MINUTE_5` via l'API Capital.com demo.

**Stratégie de pagination** :
- L'API limite le nombre de candles par appel
- On pagine **jour par jour**, fenêtres de 45 min couvrant la session DAX
- Session DAX : 09:00–17:30 CET → on collecte **07:00–16:35 UTC** pour couvrir heure d'été et d'hiver

**Output** : `data/dax_live_5min.csv`

## 1. Imports

In [1]:
import sys
import os
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from datetime import datetime, timedelta, timezone

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

from src.service.collector.session_manager import SessionManager
from src.service.collector.api_client import CapitalClient
from src.service.collector.parse import parse_candles as normalize_candles
from src.service.collector.db_importer import import_candles

print(f"Imports OK — ROOT = {ROOT}")

Imports OK — ROOT = c:\Users\Benkadda\Desktop\ENSAI\2A\S2\Python pour la datascience\Projet-INFO-2AS2


## 2. Chargement des credentials (.env)

Le fichier `.env` doit contenir :
```
CAPITAL_API_KEY=...
CAPITAL_IDENTIFIER=...
CAPITAL_PASSWORD=...
```

In [2]:
load_dotenv(ROOT / ".env")

API_KEY    = os.getenv("CAPITAL_API_KEY")
IDENTIFIER = os.getenv("CAPITAL_IDENTIFIER")
PASSWORD   = os.getenv("CAPITAL_PASSWORD")

assert API_KEY and IDENTIFIER and PASSWORD, (
    "Variables manquantes dans .env : CAPITAL_API_KEY, CAPITAL_IDENTIFIER, CAPITAL_PASSWORD"
)
print("Credentials chargés")

Credentials chargés


## 3. Connexion à l'API

In [3]:
session = SessionManager(API_KEY, IDENTIFIER, PASSWORD, ping=True)  # ping=True garde la session active
client  = CapitalClient(session)
print("Client prêt")

Session ouverte avec succès
Client prêt


## 4. Paramètres de collecte

In [4]:
EPIC       = "DE40"
RESOLUTION = "MINUTE_5"
WINDOW_MIN = 45

# 07:00–16:35 UTC couvre la session DAX en heure d'été et d'hiver
SESSION_START = "07:00"
SESSION_END   = "16:35"

SLEEP_BETWEEN_CALLS = 0.3

end_date   = datetime.now(timezone.utc).date()
start_date = end_date - timedelta(days=380)

business_days = pd.bdate_range(start=start_date, end=end_date).date.tolist()
business_days = list(reversed(business_days))

print(f"Business days à collecter : {len(business_days)}")
print(f"Période : {business_days[-1]} → {business_days[0]}")

Business days à collecter : 271
Période : 2025-04-07 → 2026-04-20


## 5. Fonction de parsing des candles

In [5]:
def parse_candles_df(prices: list) -> pd.DataFrame:
    """Convertit la liste de candles API en DataFrame. Utilise mid = (bid+ask)/2."""
    rows = []
    for p in prices:
        ts = pd.Timestamp(p["snapshotTimeUTC"], tz="UTC")
        rows.append({
            "datetime":  ts,
            "open":  (p["openPrice"]["bid"]  + p["openPrice"]["ask"])  / 2,
            "high":  (p["highPrice"]["bid"]  + p["highPrice"]["ask"])  / 2,
            "low":   (p["lowPrice"]["bid"]   + p["lowPrice"]["ask"])   / 2,
            "close": (p["closePrice"]["bid"] + p["closePrice"]["ask"]) / 2,
            "volume": p.get("lastTradedVolume", 0),
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df.set_index("datetime", inplace=True)
        df.sort_index(inplace=True)
    return df

print("Fonctions parse_candles_df et import_candles prêtes")

Fonction parse_candles définie


## 6. Collecte paginée

In [6]:
all_frames      = []
all_candles_db  = []
total_calls     = 0
total_bars      = 0

for day in business_days:
    day_str = str(day)
    session_start = datetime.strptime(f"{day_str}T{SESSION_START}:00", "%Y-%m-%dT%H:%M:%S")
    session_end   = datetime.strptime(f"{day_str}T{SESSION_END}:00",   "%Y-%m-%dT%H:%M:%S")

    day_frames  = []
    window_from = session_start

    while window_from < session_end:
        window_to = min(window_from + timedelta(minutes=WINDOW_MIN), session_end)
        from_str  = window_from.strftime("%Y-%m-%dT%H:%M:%S")
        to_str    = window_to.strftime("%Y-%m-%dT%H:%M:%S")

        try:
            result = client.get_candles_range(
                epic=EPIC, from_dt=from_str, to_dt=to_str, resolution=RESOLUTION
            )
            prices = result.get("prices", [])
        except ValueError as e:
            err_str = str(e)
            if "404" in err_str or "daterange" in err_str:
                prices = []
            else:
                print(f"Erreur {day_str} {from_str}: {e}")
                prices = []

        total_calls += 1
        if prices:
            day_frames.append(parse_candles_df(prices))
            all_candles_db.extend(normalize_candles(prices, EPIC, RESOLUTION))
            total_bars += len(prices)

        window_from = window_to
        time.sleep(SLEEP_BETWEEN_CALLS)

    if day_frames:
        all_frames.append(pd.concat(day_frames))
        print(f"  {day_str} : {sum(len(f) for f in day_frames)} bars")
    else:
        print(f"  {day_str} : fermé / aucune donnée")

print(f"\nTotal appels : {total_calls}")
print(f"Total bars   : {total_bars}")

  2026-04-20 : 128 bars
  2026-04-17 : 128 bars
  2026-04-16 : 128 bars
  2026-04-15 : 128 bars
  2026-04-14 : 118 bars
  2026-04-13 : 128 bars
  2026-04-10 : 128 bars
  2026-04-09 : 128 bars
  2026-04-08 : 128 bars
  2026-04-07 : 128 bars
  2026-04-06 : 128 bars
  2026-04-03 : 83 bars
  2026-04-02 : 128 bars
  2026-04-01 : 128 bars
  2026-03-31 : 128 bars
  2026-03-30 : 128 bars
  2026-03-27 : 128 bars
  2026-03-26 : 128 bars
  2026-03-25 : 128 bars
  2026-03-24 : 128 bars
  2026-03-23 : 126 bars
  2026-03-20 : 128 bars
  2026-03-19 : 128 bars
  2026-03-18 : 128 bars
  2026-03-17 : 128 bars
  2026-03-16 : 128 bars
  2026-03-13 : 128 bars
  2026-03-12 : 128 bars
  2026-03-11 : 128 bars
  2026-03-10 : 128 bars
  2026-03-09 : 128 bars
  2026-03-06 : 128 bars
  2026-03-05 : 128 bars
  2026-03-04 : 128 bars
  2026-03-03 : 128 bars
  2026-03-02 : 128 bars
  2026-02-27 : 128 bars
  2026-02-26 : 128 bars
  2026-02-25 : 128 bars
  2026-02-24 : 128 bars
  2026-02-23 : 128 bars
  2026-02-20 : 12

## 7. Assemblage et sauvegarde

In [7]:
if not all_frames:
    raise RuntimeError("Aucune donnée collectée — vérifier les credentials et l'epic")

df = pd.concat(all_frames)
df = df[~df.index.duplicated(keep="first")]
df.sort_index(inplace=True)

# Convertir en heure de Berlin (CET/CEST)
df.index = df.index.tz_convert("Europe/Berlin")

print(f"Total candles : {len(df):,}")
print(f"Période       : {df.index.min()} → {df.index.max()}")

output_path = ROOT / "data" / "dax_live_5min.csv"
df.to_csv(output_path)

print(f"\nSauvegardé : {output_path}")
print(f"Taille     : {output_path.stat().st_size / 1024 / 1024:.2f} MB")

report = import_candles(all_candles_db)
print(
    f"Insertion DB : {report['inserted']} insérées, "
    f"{report['skipped']} ignorées, {report['total']} traitées"
)

session.close()
print("Session fermée")

df.head(10)


Total candles : 31,033
Période       : 2025-04-07 09:00:00+02:00 → 2026-04-20 18:35:00+02:00

Sauvegardé : c:\Users\Benkadda\Desktop\ENSAI\2A\S2\Python pour la datascience\Projet-INFO-2AS2\data\dax_live_5min.csv
Taille     : 1.98 MB


,open,high,low,close,volume
datetime,,,,,
2025-04-07 09:00:00+02:00,19225.95,19254.45,18888.45,19004.95,1107
2025-04-07 09:05:00+02:00,19002.95,19108.95,18807.95,19049.45,1121
2025-04-07 09:10:00+02:00,19048.45,19218.45,19019.45,19134.45,911
2025-04-07 09:15:00+02:00,19135.45,19334.95,19135.45,19334.45,886
2025-04-07 09:20:00+02:00,19333.95,19368.95,19239.95,19292.95,686
2025-04-07 09:25:00+02:00,19293.95,19302.95,19157.45,19213.95,871
2025-04-07 09:30:00+02:00,19211.95,19211.95,19116.05,19187.05,604
2025-04-07 09:35:00+02:00,19187.55,19247.05,19146.05,19247.05,405
2025-04-07 09:40:00+02:00,19248.55,19326.05,19228.05,19268.05,335


## Résumé

| Item | Valeur |
|------|--------|
| Epic | DE40 (DAX 40) |
| Résolution | MINUTE_5 |
| Timezone | Europe/Berlin (CET/CEST) |
| Output | `data/dax_live_5min.csv` |

**Prochaine étape** → `00b_asrs_live.ipynb`